# Tushare Token 连通性测试

快速验证 `.env` 中的 `TUSHARE_TOKEN` 和 `TUSHARE_API_URL` 是否可用。

In [ ]:

token = "9389243e6c2db9924f00cde825037bd0148f918f98aa3b59b3cb0be5934e"
api_url = 'http://lianghua.nanyangqiankun.top' 
print(f'Token: {token[:8]}...{token[-4:]}  (len={len(token)})')
print(f'API URL: {api_url}')

In [ ]:
import tushare as ts

ts.set_token(token)
pro = ts.pro_api()
if api_url:
    pro._DataApi__http_url = api_url

# Test 1: 交易日历
try:
    df = pro.trade_cal(exchange='SSE', start_date='20260101', end_date='20260110')
    print(f'[PASS] trade_cal: {len(df)} rows')
    display(df.head())
except Exception as e:
    print(f'[FAIL] trade_cal: {e}')

In [ ]:
# Test 2: 股票基本信息
try:
    df = pro.stock_basic(exchange='', list_status='L', fields='ts_code,name,industry,list_date')
    print(f'[PASS] stock_basic: {len(df)} stocks listed')
    display(df.head())
except Exception as e:
    print(f'[FAIL] stock_basic: {e}')

In [ ]:
# Test 3: 日线行情 (伊利股份)
try:
    df = pro.daily_basic(ts_code='600887.SH', fields='ts_code,trade_date,close,pe_ttm,pb,dv_ttm')
    print(f'[PASS] daily_basic: {len(df)} rows')
    display(df.head())
except Exception as e:
    print(f'[FAIL] daily_basic: {e}')

In [ ]:
# Test 4: 财务指标 (VIP 接口)
try:
    df = pro.fina_indicator(ts_code='600887.SH', fields='ts_code,end_date,roe_waa,grossprofit_margin')
    print(f'[PASS] fina_indicator (VIP): {len(df)} rows')
    display(df.head())
except Exception as e:
    print(f'[FAIL] fina_indicator: {e}')
    print('  -> VIP 权限不足？请检查积分: https://tushare.pro/user/token')

In [ ]:
print('基础测试完成。\n')

# 港股股息数据校验 (02669.HK 中海物业)

对比 Tushare `hk_dividend`、yfinance、现金流量表三个来源，排查 DPS 数据是否一致。

In [ ]:
# Test 5: Tushare hk_fina_indicator (港股股息数据源)
try:
    df = pro.query("hk_fina_indicator", ts_code="02669.HK",
                   fields="ts_code,end_date,dps_hkd,divi_ratio")
    if df is not None and not df.empty:
        df = df.sort_values("end_date", ascending=False)
        print(f'[INFO] hk_fina_indicator: {len(df)} rows')
        display(df)
    else:
        print('[WARN] hk_fina_indicator: Empty result')
except Exception as e:
    print(f'[FAIL] hk_fina_indicator: {e}')

In [ ]:
# Test 6: yfinance dividends (对照组)
try:
    import yfinance as yf
    tk = yf.Ticker("2669.HK")
    divs = tk.dividends
    if divs is not None and len(divs) > 0:
        print(f'[INFO] yfinance dividends: {len(divs)} records')
        display(divs.tail(20))
    else:
        print('[WARN] yfinance dividends: Empty')
except Exception as e:
    print(f'[FAIL] yfinance: {e}')

In [ ]:
# Test 7: §5 已付股息 (cashflow - dividends paid)
try:
    for period in ["20251231", "20241231", "20231231", "20221231", "20211231"]:
        df = pro.hk_cashflow(ts_code="02669.HK", period=period)
        if df is not None and not df.empty:
            div_cols = [c for c in df.columns if "div" in c.lower()]
            paid = df.iloc[0].get("div_paid", None) or df.iloc[0].get("cash_div_paid", None)
            print(f'  {period}: div_paid={paid}  (columns with "div": {div_cols})')
        else:
            print(f'  {period}: Empty')
except Exception as e:
    print(f'[FAIL] hk_cashflow: {e}')
    # fallback: show all columns
    try:
        df = pro.hk_cashflow(ts_code="02669.HK", period="20241231")
        if df is not None and not df.empty:
            print("All columns:", list(df.columns))
    except:
        pass